# Code-based cryptography: Exam 2 NICOLAS MB ET BAPTISTE
February 5th, Thursday

## Duration: 2 hours and 15 minutes (no break).

## Extra time (special arrangements): + 45 minutes -> 3 hours in total.

You can write your answers on paper or directly into this notebook.

*N.B. : We number the indices starting from zero to match Sage 0-based indexing.*

In the first part of the course, we saw that the star–product provides a distinguisher between GRS
and random linear codes. A priori, this distinguisher is not yet prohibitive for the use of GRS codes in the McEliece cryptosystem. In this exercise, we will show how it can be turned into an attack to recover the secret parameters of a public code $C_{pub} = \textsf{GRS}_k(\mathbf{x}, \mathbf{y})$, i.e. the support $\mathbf{x} = (x_0, . . . , x_{n−1})$ and the multiplier $\mathbf{y} = (y_0, . . . , y_{n−1})$, given by a generator matrix $G_{pub}$.

## 1. Preliminaries 

In [117]:
#The following function return random support and multiplier
#to construct random GRS code over F_q with length n and dimension k.

def RandomSupport(F,n):
    list_x=[]
    while len(list_x)<n :
        x=F.random_element()
        if not x in list_x:
            list_x.append(x)
    return list_x

def RandomMultiplier(F,n):
    list_y=[]
    while len(list_y)<n :
        y=F.random_element()
        if y != 0:
            list_y.append(y)
    return list_y

# To compute Schur products of vectors and codes

def SchurProduct(u,v):
    n = len(u)
    if len(v) != n :
        print("Erreur : les vecteurs n'ont pas la même longueur")
        return
    return vector([u[i]*v[i] for i in range(n)])

def SchurProductCodes(U,V):
    n = U.length()
    if V.length() != n :
        print("Erreur : les deux codes n'ont pas la même longueur")
        return
    F = U.base_field() #Obtention du corps de base
    GU= U.generator_matrix()
    GV= V.generator_matrix()
    GUV=[]
    for u in GU.rows():
        for v in GV.rows():
            GUV.append(SchurProduct(u,v))
    return codes.LinearCode(matrix(F,GUV))

**Q1.a.** Let $\alpha,\gamma \in \mathbb{F}_q^*$ and $\mathbf{b}=(b,\dots,b) \in \mathbb{F}_q^n$.

Check that $\textsf{GRS}_{k}(\mathbf{x}, \mathbf{y}) = \textsf{GRS}_{k}(a\mathbf{x}+\mathbf{b}, \gamma \mathbf{y})$ 50 times for  random values of $\alpha,\gamma, \mathbf{b}, \mathbf{x}$ and $\mathbf{y}$.

You should use the function ``codes.GeneralizedReedSolomonCode(list_x, k, list_y)`` from Sage and the functions ``RandomSupport`` and ``RandomMultiplier`` above.

In [118]:
# For your tests
n=25
q=random_prime(100, lbound=n+1)
Fq=GF(q)
k=randint(4,n)

def nonnul(F):
    w=F.random_element()
    while w==F(0):
        w=F.random_element()
    return w

res=True
for _ in range(50):
    list_x=RandomSupport(Fq, n)
    list_y=RandomMultiplier(Fq, n)
    alpha=nonnul(Fq)
    gamma=nonnul(Fq)
    bj=Fq.random_element()
    b=vector(Fq,[bj for j in range(n)])
    C1=codes.GeneralizedReedSolomonCode(list_x, k, list_y)
    C2=codes.GeneralizedReedSolomonCode(vector(Fq,[alpha*list_x[j] + bj for j in range(n)]), k, vector(Fq, [gamma*list_y[j] for j in range(n)]))
    if (not C1.is_subcode(C2)) or (not C2.is_subcode(C1)):
        res = False
    
print(res)

True


**Q1.b BONUS:** Give a clean proof of this property. *ONLY IF YOU HAVE TIME!*

We **admit** that the property above is always true. Explain why we can assume that $x_0=0$, $x_1=1$ and $y_0=1$. 



On peut juste prendre gamma=1/y_0
Pour le reste on veut $ \alpha, b $ tels que $a*x_0 + b= 0$ et $a*x_1 + b = 1$
On trouve $\alpha = \frac{1}{x_1-x_0}$ et $b=\frac{-x0}{x_1-x_0}$


For the next section, you will use the code `Ctest` to test your functions.

In [119]:
F128=GF(128)
n=100
k=30

def RandomSupport01(F,n):
    list_x=[F(0),F(1)]
    while len(list_x)<n :
        x=F.random_element()
        if not x in list_x:
            list_x.append(x)
    return list_x

x=RandomSupport01(F128,n) #starts with 0 and 1
y=[1]+RandomMultiplier(F128,n-1) #starts with 1
Ctest=codes.GeneralizedReedSolomonCode(x, k, y)

#Just a sanity check: the function SchurProduct does what it is supposed to do.
#assert SchurProductCodes(Ctest,Ctest) == codes.GeneralizedReedSolomonCode(x, 2*k-1, SchurProduct(y,y)) 

## 2. Filtration
In order to recover the secret parameters, we will build a *filtration* for $C_{pub}$, *i.e.* a sequence $(C_{i})$ of codes such that
              $$C_{pub} =C_{0} \supset C_{1} \supset C_{2} \supset \dots \supset C_{i} \supset \dots \supset C_k$$
We define $C_1$ as the subcode of $C_{pub}$ of vectors whose $0^{th}$ coordinate is equal to $0$.

**Q2.a.** Prove that $C_1$ has dimension $k-1$ and consists in evaluating polynomials divisible by $X$.

On a que $$C_{\text{pub}} = \textsf{GRS}_k(\mathbf{x}, \mathbf{y}) = \text{Vect}(\{ (f(x_0)*y_0, f(x_1)*y_1, \ldots, f(x_n)*y_n) | f \in {1,X, \ldots, X^{k-1}}\})$$. 
Or, comme $x_0 = 0$ et $y_0 = 1$ et que, dans $C_1$, $f(x_0) = f(0) = 0$ alors $ X | f $. On en déduit que $f \in \{ X, \ldots, X^{k-1}\}$ et donc que $$ C_1 = \text{Vect}(\{ (f(x_0)*y_0, f(x_1)*1, \ldots, f(x_n)*y_n) | f \in \{X, \ldots, X^{k-1}\})$$ d'où $\text{dim}(C_1) = k-1$

**Q2.b.** Explain why the function `SubCode1(C)` returns the subcode $C_1$ of the input code $C$ as defined above.

In [1]:
def SubCode1(C):
    H=C.parity_check_matrix()
    F=C.base_field()
    n=C.length()
    L=matrix([F(1)]+[0 for i in range(n-1)])
    V=kernel(H.transpose())
    Z=kernel(L.transpose())
    return codes.LinearCode(V.intersection(Z))
    


Z c'est l'ensemble des matrices dont la première colonne est faite de 0.
V c'est l'ensemble des mots du code.

On a donc bien $V\cap Z = C_1$ par définition de C1


We define $C_i$ the subcode of $C_{pub}$ obtained by evaluating polynomials divisible by $X^i$ for $0 \le i \leq k-1$.

**Q3.** Let $1\le i\le k-2$. Prove that $C_{i+1} \star C_{i-1} = C_i^{(2)}$.

*Hint: You should think in terms of monomials.*

On a que 
$$ C_i = y \star \text{ev}_{x}(\text{Span}(X^i, \ldots, X^{k-1})) $$
d'où 
$$ C_i^{(2)} = (y \star y) \star \text{ev}{x}(\text{Span}(X^i, \ldots, X^{k-1}) \star \text{Span}(X^i, \ldots, X^{k-1})) = (y \star y)\star \text{Span}(\{X^i \times X^j\}{i \leq l \leq m \leq k-1 }) $$

et

$$C_{i+1} \star C_{i-1} = (y \star y) \star \text{ev}{x}(\text{Span}(X^{i+1}, \ldots, X^{k-1}) \star \text{Span}(X^{i-1}, \ldots, X^{k-1})) = (y \star y)\star \text{Span}(\{X^i \times X^j\}{(i+1 \leq l \leq k-1),(i-1 \leq m \leq k-1)}).$$

Or, 
$$ \{X^i \times X^j \}{i \leq l \leq m \leq k-1 }  = \{X^i \times X^j\}{(i+1 \leq l \leq k-1),(i-1 \leq m \leq k-1)} $$

On en déduit l'égalité des deux codes, $C_i^{(2)} = C_{i+1} \star C_{i-1}$

We admit $C_{i+1}=\{\mathbf{c} \in C_i \mid \mathbf{c} \star C_{i-1} \subseteq C_i^{(2)}\}$.

                
**Q4.** We recall that any pair of codes $A$ and $B$ of $\mathbb{F}_q^n$,
                $$(A \star B^\perp)^\perp=\{\mathbf{z} \in \mathbb{F}_q^n \mid \mathbf{z} \star A \subseteq B\}.$$
*Do not prove this result again.*

Using this result, write a function ``ProductInSquare(U,V)`` which takes as inputs two codes $U, V \subset \mathbb{F}_q^n$ and returns the code $W=\{ \mathbf{v} \in V \mid \mathbf{v} \star U \subseteq V^{(2)}\}$.

*N.B.: You are invited to take inspiration from the function `SubCode1` to design the function `ProductInSquare`.*

In [121]:


def ProductInSquare(U,V):
    V2=SchurProductCodes(V,V)
    dualprod=SchurProductCodes(U,V2.dual_code()).dual_code()
    H1=dualprod.parity_check_matrix()
    H2=V.parity_check_matrix()
    Z1=kernel(H1.transpose())
    Z2=kernel(H2.transpose())
    return codes.LinearCode(Z1.intersection(Z2))



C1=SubCode1(Ctest)
C2=ProductInSquare(Ctest,C1)
print(Ctest)
print(C1)
print(C2)

[100, 30, 71] Generalized Reed-Solomon Code over GF(128)
[100, 29] linear code over GF(128)
[100, 28] linear code over GF(128)


**Q5.a.** Use the previous question to write a function ``Filtration(C)`` with inputs an $[n,k]$ code $C$ and returns the sequence of codes $C_i$ for $1 \leq i \leq k-1$.

In [122]:
def Filtration(C):
    F=C.base_field()
    n=C.length()
    k=C.dimension()
    C0=C
    listdeC=[C0]
    C1=SubCode1(C0)
    listdeC.append(C1)
    for i in range(2, k):
        Ci=ProductInSquare(listdeC[i-2],listdeC[i-1])
        listdeC.append(Ci)
    return listdeC
    

**Q5.b.** Check that the filtration you have built above is *indeed* a filtration, i.e. $C_i \star C_j \subseteq C_{i+j}$ for $1 \leq i,j \leq k-1$ using `SchurProductCodes`.

If it is not the case, please *carefully* read Q4 again. The issue is likely to be in your definition of `ProductInSquare`.

In [123]:


def ExternSquareProduct(v, C):
    G=C.basis()
    n = len(G)
    Gresult = []
    for i in range (n):
        vec= SchurProduct(G[i], v)
        Gresult.append(vec)
    Gresult = Matrix(Gresult)
    CstarC=codes.LinearCode(Gresult)
    return CstarC

listtest=Filtration(Ctest)
res=True
for _ in range(20):
    i=randint(1, floor((k-1)/2))
    j=randint(1, floor((k-1)/2))
    prod=SchurProductCodes(listtest[i],listtest[j])
    ylisttestij = ExternSquareProduct(y,listtest[i+j])
    if not ylisttestij.is_subcode(prod):

        res=False
        print(i)
        print(j)
        print(prod)
        print(listtest[i+j]) 
print(res)




True


**Q6.** What is the dimension of $C_{k-1}$? What is the shape of a basis ?

On décroit de une dimension à chaque fois donc c'est de dimension 1 et sa base est de taille (1, n)

Check your answers using the output of ``Filtration(Ctest)``.

In [124]:
print(Filtration(Ctest)[k-1])

[100, 1] linear code over GF(128)


**Q7.a.** Now consider the subcode $C'_{k-2} = \{ \mathbf{c}=(c_0,\dots,c_{n-1}) \in C_{k-2} \mid c_1=0 \} \subset C_{k-2}$.
Adapt the function `Subcode1` given in **Q2.b.** to compute $C'_{k-2}$ from $C_{k-2}$.

In [125]:
def SubCode2(C):
    H=C.parity_check_matrix()
    F=C.base_field()
    n=C.length()
    L=matrix([0]+[F(1)]+[0 for i in range(n-2)])
    V=kernel(H.transpose())
    Z=kernel(L.transpose())
    return codes.LinearCode(V.intersection(Z))

Cprimekmoins2 = SubCode2(listtest[k-2])


**Q7.b.** What is the dimension of $C'_{k-2}$? What is the shape of a basis?

Normalement on avait un truc de dimension 2 mais là on a viré une dimension en plus donc on est de dimension 1 et donc toujours une base (1,n)

Check your answers in practive.

In [126]:
print(Cprimekmoins2.dimension())

1


**Q8.** Using the assumption $y_0=1$, explain how the support $\mathbf{x}$ can be easily recovered from bases of $C_{k−1}$ and $C'_{k−2}$.

Voir feuille 

Write a piece of code that computes the support given `Filtration(Ctest)`. Check your result by comparing with the actual support `x`.

In [127]:
def trouvex(C):
    k=C.dimension()
    n=C.length()
    listedeC=Filtration(Ctest)
    Ckmoins1=listedeC[k-1]
    Ckprimemoins2=SubCode2(listedeC[k-2])
    B1=Ckmoins1.basis()
    vec1=B1[0]
    B2=Ckprimemoins2.basis()
    vec2=B2[0]
    print(vec1)
    print(vec2)
    x=[0, 1]
    x2=2#En vrai il faut bruteforce...
    x.append(x2)
    for i in range (3,n):
        xi=1/(1-(vec2[i]/vec1[i])*(vec1[i-1]/vec2[i-1])*(1-1/x[i-1]))
    

trouvex(Ctest)

(0, 1, z7^5 + z7^2, z7^6 + z7^4 + z7, z7^5 + z7^4 + 1, z7^5 + z7^4 + z7 + 1, z7^4 + z7^3, z7^5 + z7^2, z7^6 + z7^4 + z7^3 + z7^2 + z7, z7^5 + z7^4 + z7^3 + z7, z7^6 + z7^4 + z7^3 + 1, z7^4 + z7^2 + 1, z7^5, z7^4 + z7^3 + 1, z7^6 + z7^5 + z7^2, z7^6 + z7^2 + z7, z7^3 + z7, z7^5 + z7^3 + z7^2 + z7, z7^6 + z7^5 + z7^4 + z7^2 + z7 + 1, z7^6 + z7^5 + z7^4 + 1, z7^5 + z7^3 + z7^2 + z7 + 1, z7^6 + z7^5 + z7^2 + z7, z7^5 + z7^4 + z7^3 + z7, z7^3 + z7^2 + z7, z7^6 + z7^5 + z7^4 + z7^2, z7^6 + z7^4 + z7^2 + z7, z7^4 + 1, z7^6 + z7^5 + z7^2 + 1, z7^5 + z7, z7^6 + z7^5 + z7^3 + z7^2 + 1, z7^5 + z7^3 + z7^2, z7^6 + z7^4 + z7^3 + z7^2, z7^4, z7^5 + z7^3 + z7, z7^4 + z7^3 + z7^2 + 1, z7^6 + z7^4 + z7^2 + z7 + 1, z7^6 + z7^4 + z7^2, z7^6 + z7^5 + z7^4 + z7 + 1, z7^6 + z7^2, z7^5 + z7^4 + z7^3 + z7^2 + z7, z7^6 + z7^4 + z7^2, z7^6 + z7^5 + z7^4 + z7^3 + z7, z7^6 + z7^2 + 1, z7^5 + z7^4 + z7^3 + z7^2 + z7 + 1, z7^5 + z7^2 + z7, z7^5 + z7^3 + z7^2 + z7 + 1, z7^5 + 1, z7^6 + z7^5 + z7^4 + z7, z7^6 + z7^3 

TypeError: unsupported operand parent(s) for *: 'Finite Field in z7 of size 2^7' and 'Rational Field'

**Q9.** Knowing the support $\mathbf{x}$, explain how you can now recover the multiplier $\mathbf{y}$. 

Voir feuille...

Write a piece of code that achieves this and compare with the actual multiplier `y`.

**Q10.** Analyse the complexity of `ProductInSquare`to give the complexity of retrieving the support and the multiplier of an unkown GRS code using this filtration attack.

In [ ]:
#Write your answer here or on paper.